# LSTM (TensorFlow)

Thesis experiment: two stacked LSTM layers, rolling-origin evaluation, three feature sets. Needs `tensorflow`.

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Flatten

In [ ]:
from energyforecast.data import load_dataset

data = load_dataset("../../data")

## Model

In [ ]:
def create_sliding_window(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

In [ ]:
def plot_model_rmse_and_loss(history):

    #evaluating train and validation accuracies and losses

    train_rmse = history.history['root_mean_squared_error']
    #val_rmse = history.history['val_root_mean_squared_error']

    train_loss = history.history['loss']
    #val_loss = history.history['val_loss']

    #visualizing epochs vs. train and validation accuracies and losses

    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.plot(train_rmse, label='Training RMSE')
    #plt.plot(val_rmse, label='Validation RMSE')
    plt.legend()
    plt.title('Epochs vs. Training and Validation RMSE')

    plt.subplot(1, 2, 2)
    plt.plot(train_loss, label='Training Loss')
    #plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Epochs vs. Training and Validation Loss')

    plt.show()

def plot_preds_vs_actual(true, preds):
    plt.figure(figsize=(12,6))
    plt.plot(true, label='Real')
    plt.plot(preds, label='Predicted', color = 'red')
    plt.title('Predicted vs Real Values')
    plt.title('Actual vs Predicted Values')
    plt.xlabel('Time')
    plt.ylabel('Total Aggregated')
    plt.legend()
    plt.show()

In [ ]:
#train_split = 0.7
n_train = 35064
n_test = int(len(data)-n_train)
#sequence_length = 24

features = ['total_aggregated']
n_features = len(features)
feature_array = data[features].values

# (2)
# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train].reshape(-1,1))

# Transform both Training and Test data
scaled_array = target_scaler.transform(feature_array)

X_train, y_train = create_sliding_window(scaled_array[:n_train], 1)
X_test, y_test = create_sliding_window(scaled_array[n_train:], 1)

In [ ]:
truth = feature_array[-len(y_test):]

In [ ]:
input_shape1 = X_train.shape[-2:]

loss = tf.keras.losses.MeanSquaredError()

metric = [tf.keras.metrics.RootMeanSquaredError()]

#lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4 * 10**(epoch / 10))

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=0.003)

In [ ]:
tf.keras.backend.clear_session()

#model building

lstm = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, return_sequences=True, input_shape=input_shape1),
    tf.keras.layers.LSTM(10, return_sequences=True),
    tf.keras.layers.Flatten(),
    #tf.keras.layers.Dense(1, activation='relu'),
    tf.keras.layers.Dense(1),
])

lstm.summary()

lstm.compile(loss=loss, optimizer=optimizer, metrics=metric)

In [ ]:
n_forecast_steps = 24  # forecast horizon
window = 24*365*3

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions = []  # forecasts
errors = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    lstm.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = lstm.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    # rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    #errors.append(rmse)
    predictions.extend(yhat_rescaled)

    #print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
#y_true = pd.Series(feature_array[-len(predictions):].flatten())
res=pd.DataFrame()
res['y_true'] = pd.Series(truth.flatten())
res['y_pred'] = predictions

rmse = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('RMSE:', rmse)

In [ ]:
plot_preds_vs_actual(truth, predictions)

## Weekend dummies

In [ ]:
#train_split = 0.7
n_train = 35064
n_test = int(len(data)-n_train)

features = ['total_aggregated']
sat = ['saturday']
sun = ['sunday']

feature_array = data[features].values
#dummy = data[we].values
dummy_1 = data[sat].values
dummy_2 = data[sun].values
dummies = [dummy_1, dummy_2]

## concatenate dummy variables with feature_array
#feature_array = np.concatenate((feature_array, dummy_array), axis=1)

In [ ]:
# Fit Scaler only on Training target values
target_scaler = StandardScaler()

# Transform both Training and Test data
target_scaler.fit(feature_array[:n_train].reshape(-1,1))
scaled_array = target_scaler.transform(feature_array)

In [ ]:
# concatenate dummy variables with feature_array
final_array = np.concatenate((scaled_array, dummy_1), axis=1)
final_array = np.concatenate((final_array, dummy_2), axis=1)

In [ ]:
sequence_length = 1
#n_features = 3
X, y = create_sliding_window(final_array, 1)

In [ ]:
# Then, split into train and test
X_train = X[:n_train]
y_train = y[:n_train, 0]  # Select only the first column (the scaled feature values)

X_test = X[n_train:]
y_test = y[n_train:, 0]  # Select only the first column (the scaled feature values)

In [ ]:
input_shape2 = X_train.shape[-2:]

In [ ]:
tf.keras.backend.clear_session()

#model building

lstm2 = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, return_sequences=True, input_shape=input_shape2),
    tf.keras.layers.LSTM(10, return_sequences=True),
    tf.keras.layers.Flatten(),
    #tf.keras.layers.Dense(1, activation='relu'),
    tf.keras.layers.Dense(1),
])

lstm2.summary()

lstm2.compile(loss=loss, optimizer=optimizer, metrics=metric)

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions2 = []  # forecasts
errors2 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    lstm2.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = lstm2.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors.append(rmse)
    predictions2.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
#truth2 = pd.Series(feature_array[-len(predictions2):].flatten())
#res2=pd.DataFrame()
#res2['y_true'] = truth2
res['y_pred2'] = predictions2

In [ ]:
rmse2 = sqrt(np.mean((res.y_true - res.y_pred2)**2))
print('RMSE:', rmse2)

In [ ]:
plot_preds_vs_actual(res.y_true, res.y_pred2)

## Business-hour dummy

In [ ]:
# 2020 starts at index 35064

features = ['total_aggregated']
dummy_features = ['business_hour']

feature_array = data[features].values
dummy_array = data[dummy_features].values

## concatenate dummy variables with feature_array
#feature_array = np.concatenate((feature_array, dummy_array), axis=1)

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train].reshape(-1,1))

# Transform both Training and Test data
scaled_array = target_scaler.transform(feature_array)

# concatenate dummy variables with feature_array
final_array = np.concatenate((scaled_array, dummy_array), axis=1)

#sequence_length = 2
#X_train, y_train = create_sliding_window(final_array[:n_train], sequence_length)
#X_test, y_test = create_sliding_window(final_array[n_train:], sequence_length)

In [ ]:
#sequence_length = 24
#n_features = 3
X, y = create_sliding_window(final_array, 1)
# Then, split into train and test
X_train = X[:n_train]
y_train = y[:n_train, 0]  # Select only the first column (the scaled feature values)

X_test = X[n_train:]
y_test = y[n_train:, 0]  # Select only the first column (the scaled feature values)

In [ ]:
input_shape3 = X_train.shape[-2:]

tf.keras.backend.clear_session()

In [ ]:
#model building

lstm3 = tf.keras.Sequential([
    tf.keras.layers.LSTM(10, return_sequences=True, input_shape=input_shape3),
    tf.keras.layers.LSTM(10, return_sequences=True),
    tf.keras.layers.Flatten(),
    #tf.keras.layers.Dense(1, activation='relu'),
    tf.keras.layers.Dense(1),
])

lstm3.summary()

lstm3.compile(loss=loss, optimizer=optimizer, metrics=metric)

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions3 = []  # forecasts
errors3 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    lstm3.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = lstm3.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors.append(rmse)
    predictions3.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
#res3=pd.DataFrame()
#res3['y_true'] = res['y_true']
res['y_pred3'] = predictions3
#scaled_predictions = target_scaler.transform(predictions3)
#scaled_truth = target_scaler.transform(truth.reshape(-1,1))
#
rmse3 = sqrt(np.mean((res.y_true - res.y_pred3)**2))
print('Root Mean Squared Error:', rmse3)

In [ ]:
plot_preds_vs_actual(res.y_true, res.y_pred3)

In [ ]:
rmse1 = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('Root Mean Squared Error just TS:', rmse1)
rmse2 = sqrt(np.mean((res.y_true - res.y_pred2)**2))
print('Root Mean Squared Error WE dummies:', rmse2)
rmse3 = sqrt(np.mean((res.y_true - res.y_pred3)**2))
print('Root Mean Squared Error BH dummy:', rmse3)